In [2]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

ROOT  = Path().resolve().parent
RAW   = ROOT / 'data' / 'processed' / 'substations'
PROC  = ROOT / 'data' / 'processed' / 'substations'

# Processed interchange (fromba always in CA8)
print('Loading processed substation load profiles ...')
loads = pd.read_csv(PROC / 'substation_load_profiles.csv', low_memory=False)
print(f'  {len(loads):,} load rows')
# Region TI
print('Loading locations data ...')
locations = pd.read_csv(RAW / 'substation_locations.csv')
print(f'  {len(locations):,} location rows')

Loading processed substation load profiles ...
  455,568 load rows
Loading locations data ...
  2,614 location rows


In [3]:
locations.head()

,utility,substation_name,latitude,longitude,voltage_kv,substation_type,sys_name,division,subst_id,existing_gen,...,ind_pct,other_pct,res_total,com_total,agr_total,ind_total,other_total,note_sub,existing_der,net_min_daytime_load_mw
0,pge,ALMADEN,37.260104,-121.889833,12.0,NaN,NaN,San Jose,8231.0,7.152,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN
1,pge,HOPLAND,38.972880,-123.081171,12.0,NaN,NaN,Humboldt,4225.0,4.324,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN
2,pge,KEARNEY,36.706696,-119.880091,12.0,NaN,NaN,Fresno,25270.0,2.751,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN
3,pge,STAGG,37.991681,-121.347884,21.0,NaN,NaN,Stockton,16242.0,5.083,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN
4,pge,BURNEY,40.888216,-121.670840,12.0,NaN,NaN,North Valley,10331.0,0.782,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN


In [4]:
locations['voltage_kv'].unique()

array([12.  , 21.  ,  4.16, 17.  , 12.55,   nan, 16.98,  2.5 , 34.5 ,
        5.  ,  6.88, 25.  ])

In [5]:
locations.loc[locations['existing_gen']<0.001]#,'utility'].unique()

,utility,substation_name,latitude,longitude,voltage_kv,substation_type,sys_name,division,subst_id,existing_gen,...,ind_pct,other_pct,res_total,com_total,agr_total,ind_total,other_total,note_sub,existing_der,net_min_daytime_load_mw
93,pge,PITTSBURG,38.030707,-121.869259,4.16,NaN,NaN,Diablo,1216.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Yes,NaN,NaN
119,pge,WELLFIELD,35.224963,-118.778337,12.00,NaN,NaN,Kern,25429.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN
140,pge,WOODLAND,38.678361,-121.759213,12.00,NaN,NaN,Sacramento,6203.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Yes,NaN,NaN
214,pge,PINECREST,38.187947,-119.993800,4.16,NaN,NaN,Yosemite,16316.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN
428,pge,BALCH NO 1,36.908322,-119.086228,12.00,NaN,NaN,Fresno,25250.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1292,sce,Tejon Peak P.T.,34.787468,-118.819148,2.50,NaN,Bailey 220/66 System,NaN,1574.0,0.0,...,0.0,33.3333,0.0,4.0,0.0,0.0,2.0,Interconnection studies in this area have iden...,NaN,NaN
1347,sce,Weesha P.T.,34.174853,-116.948896,2.50,NaN,El Casco 220/115 System,NaN,1763.0,0.0,...,0.0,32.3529,21.0,0.0,2.0,0.0,11.0,Deliverability in this area has not been deter...,NaN,NaN
1355,sce,White Mt.,37.411111,-118.187107,12.55,NaN,Inyo Sce 220/115 System,NaN,206.0,0.0,...,0.0,50.0000,0.0,2.0,0.0,0.0,2.0,Interconnection studies in this area have iden...,NaN,NaN
1364,sce,Yankee P.T.,35.623319,-118.418387,2.50,NaN,Vestal 220/66 System,NaN,1605.0,0.0,...,0.0,33.3333,0.0,2.0,0.0,0.0,1.0,Interconnection studies in this area have iden...,NaN,NaN


In [6]:
locations.loc[locations['substation_name']=='PITTSBURG']

,utility,substation_name,latitude,longitude,voltage_kv,substation_type,sys_name,division,subst_id,existing_gen,...,ind_pct,other_pct,res_total,com_total,agr_total,ind_total,other_total,note_sub,existing_der,net_min_daytime_load_mw
93,pge,PITTSBURG,38.030707,-121.869259,4.16,NaN,NaN,Diablo,1216.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Yes,NaN,NaN


In [7]:
print(locations.loc[locations['utility'] == 'pge', 'existing_gen'].min())
print(locations.loc[locations['utility'] == 'pge', 'existing_gen'].max())

0.0
70.142


In [8]:
pge_attrs_path = ROOT / "data/raw/pge/pge_substation_attributes.csv"
pge_loads_path = ROOT / "data/raw/pge/pge_layer25_earliest_latest_part001.csv"
pge_attrs = pd.read_csv(pge_attrs_path)
pge_loads = pd.read_csv(pge_loads_path)
pge_bad_stations = pge_attrs.loc[(~pge_attrs['substation_name'].isin(pge_loads['subname']))].copy(deep=True)
print(pge_bad_stations.shape)
pge_bad_stations2 = pge_loads.loc[(~pge_loads['subname'].isin(pge_attrs['substation_name']))].copy(deep=True)
print(pge_bad_stations2.shape)

(42, 12)
(0, 10)


In [9]:
sce_loads_path = ROOT / "data/raw/sce/sce_bulk_download_all.csv"
sce_attrs_path= ROOT / "data/raw/sce/sce_substation_attributes.csv"
sce_load = pd.read_csv(sce_loads_path)
sce_attrs = pd.read_csv(sce_attrs_path)
sce_bad_stations = sce_load.loc[(~sce_load['SUBSTATION'].str.lower().str.replace(' p.t.','').isin(sce_attrs['substation_name'].str.lower().str.replace(' p.t.',''))),'SUBSTATION'].unique()
print(sce_bad_stations.shape)
sce_bad_stations2 = sce_attrs.loc[(~sce_attrs['substation_name'].str.lower().str.replace(' p.t.','').isin(sce_load['SUBSTATION'].str.lower().str.replace(' p.t.','')))].copy(deep=True)
print(sce_bad_stations2.shape)

(0,)
(507, 22)


In [10]:
sce_load['SUBSTATION'].unique()
sce_attrs['substation_name'].unique()

<StringArray>
['Acequia P.t.',        'Acton',      'Aerojet',          'Afg',
     'Aha P.t.',      'Airchem',     'Alamitos',        'Alamo',
  'Albury P.t.',        'Alder',
 ...
  'Yankee P.t.',        'Yermo',  'Yorba Linda',      'Yucaipa',
        'Yucca',        'Yukon',         'Zack',        'Zanja',
   'Ziggy P.t.',     'Zondwind']
Length: 1220, dtype: str

In [11]:
sce_bad_stations2.drop_duplicates(subset=['substation_name']).to_csv('sce_stations_no_load.csv')

In [12]:
sce_bad_stations2['substation_name']

0       Acequia P.t.
2            Aerojet
3                Afg
5            Airchem
6           Alamitos
            ...     
1195           Wharf
1200      Willamette
1203         Windhub
1204    Winding P.t.
1219        Zondwind
Name: substation_name, Length: 507, dtype: str

In [13]:
sce_attrs.loc[(sce_attrs['existing_gen']==0)&((sce_attrs['queued_gen']==0))&(sce_attrs['total_gen']==0)].to_csv('sce_zero_gen.csv')

In [14]:
sce_bad_stations2.loc[sce_bad_stations2['substation_name'].duplicated()]

,substation_name,subst_id,sys_name,existing_gen,queued_gen,total_gen,projected_load,der_penetration,max_remain_cap,voltage_kv,...,com_pct,agr_pct,ind_pct,other_pct,res_total,com_total,agr_total,ind_total,other_total,note_sub


In [15]:
unique_dict = {col: len(sce_bad_stations2[col].unique()) for col in sce_bad_stations2.columns}
print(unique_dict)

{'substation_name': 507, 'subst_id': 446, 'sys_name': 60, 'existing_gen': 76, 'queued_gen': 58, 'total_gen': 76, 'projected_load': 228, 'der_penetration': 51, 'max_remain_cap': 178, 'voltage_kv': 7, 'circuit_count': 10, 'res_pct': 44, 'com_pct': 43, 'agr_pct': 7, 'ind_pct': 8, 'other_pct': 51, 'res_total': 40, 'com_total': 32, 'agr_total': 6, 'ind_total': 5, 'other_total': 55, 'note_sub': 4}


In [16]:
sdge_load.head()

NameError: name 'sdge_load' is not defined

In [17]:
sdge_loads_path = ROOT / "data/raw/sdge/sdge_substation_profiles_part001.csv"
sdge_attrs_path= ROOT / "data/raw/sdge/sdge_substation_attributes.csv"
sdge_load = pd.read_csv(sdge_loads_path)
sdge_attrs = pd.read_csv(sdge_attrs_path)
sdge_bad_stations = sdge_load.loc[~sdge_load['AssetName'].str.upper().isin(sdge_attrs['substation_name'])].copy(deep=True)
print(sdge_bad_stations.shape)
sdge_bad_stations2 = sdge_attrs.loc[~sdge_attrs['substation_name'].isin(sdge_load['AssetName'].str.upper())].copy(deep=True)
print(sdge_bad_stations2.shape)

(0, 32)
(8, 10)


In [18]:
sdge_bad_stations2

,substation_name,substation_type,voltage_kv,existing_gen,queued_gen,total_gen,projected_load,der_penetration,longitude,latitude
3,KYOCERA,69/12 kV,12kV,3.848,0.000,3.848,NaN,31,-117.140351,32.819687
15,PENDLETON,69/12 kV,12kV,9.670,0.000,9.670,NaN,16,-117.311779,33.308408
37,MIRA SORRENTO,69/12 kV,12kV,16.314,1.080,17.394,NaN,28,-117.213847,32.906143
40,LOVELAND,69/12 kV,12kV,3.276,12.651,15.927,NaN,11,-116.778580,32.786284
64,BASILONE,69/12 kV,12kV,3.853,0.000,3.853,NaN,14,-117.582295,33.387681
77,MIRAMAR,69/12 kV,12kV,17.473,0.254,17.726,NaN,20,-117.142186,32.894149
93,DIVISION,69/12 kV,12kV,0.000,NaN,NaN,NaN,0,-117.118295,32.678503
97,WARREN CANYON,69/12/4 kV,12kV,0.000,NaN,NaN,NaN,0,-117.000651,33.020611


In [19]:
sdge_attrs.loc[sdge_attrs['existing_gen']==0]

,substation_name,substation_type,voltage_kv,existing_gen,queued_gen,total_gen,projected_load,der_penetration,longitude,latitude
93,DIVISION,69/12 kV,12kV,0.0,NaN,NaN,NaN,0,-117.118295,32.678503
97,WARREN CANYON,69/12/4 kV,12kV,0.0,NaN,NaN,NaN,0,-117.000651,33.020611


In [34]:
calSubs = pd.read_csv(ROOT / 'data/processed/substation_misc/ca_substations_2022.csv')

In [35]:
calSubs = calSubs.loc[calSubs['type']=='SUBSTATION'].copy(deep=True)

In [36]:
calSubs.head()

,name,owner_raw,owner_std,type,hifld_id,max_voltage_kv,latitude,longitude,county,city,zip_code,source,path
0,Jenney,Other,other,SUBSTATION,310025.0,115.0,37.772436,-122.242916,Alameda County,Alameda,94501,CEC,NaN
1,Corona,PG&E,pge,SUBSTATION,306474.0,115.0,38.265013,-122.657369,Sonoma County,Petaluma,94954,CEC,NaN
2,South Bay 1,PG&E,pge,SUBSTATION,310124.0,NaN,37.778482,-121.625951,Alameda County,Unincorporated,94514,CEC,NaN
3,Altamont,PG&E,pge,SUBSTATION,306256.0,60.0,37.748377,-121.677208,Alameda County,Unincorporated,94551,CEC,NaN
4,Castro Valley,PG&E,pge,SUBSTATION,303892.0,230.0,37.691281,-122.061277,Alameda County,Unincorporated,94546,CEC,NaN


In [37]:
len(calSubs)

3371

In [38]:
calSubsPGE = calSubs.loc[(calSubs['owner_raw']=='PG&E')&(calSubs['name'].str.lower().isin(pge_loads['subname'].str.lower()))].copy(deep=True)
calSubsNotPGE = calSubs.loc[(calSubs['owner_raw']=='PG&E')&(~calSubs['name'].str.lower().isin(pge_loads['subname'].str.lower()))].copy(deep=True)

In [42]:
len(pge_loads['subname'].unique())

664

In [40]:
calSubsPGE

,name,owner_raw,owner_std,type,hifld_id,max_voltage_kv,latitude,longitude,county,city,zip_code,source,path
1,Corona,PG&E,pge,SUBSTATION,306474.0,115.0,38.265013,-122.657369,Sonoma County,Petaluma,94954,CEC,NaN
4,Castro Valley,PG&E,pge,SUBSTATION,303892.0,230.0,37.691281,-122.061277,Alameda County,Unincorporated,94546,CEC,NaN
8,Dumbarton,PG&E,pge,SUBSTATION,307780.0,115.0,37.556992,-122.057940,Alameda County,Fremont,94555,CEC,NaN
10,Edes,PG&E,pge,SUBSTATION,303133.0,115.0,37.738690,-122.193799,Alameda County,Oakland,94603,CEC,NaN
11,Fremont,PG&E,pge,SUBSTATION,302108.0,115.0,37.541363,-121.961957,Alameda County,Fremont,94538,CEC,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2939,Orick,PG&E,pge,SUBSTATION,303172.0,60.0,41.294287,-124.056871,Humboldt County,Unincorporated,95555,CEC,NaN
2946,San Miguel,PG&E,pge,SUBSTATION,303705.0,60.0,35.752336,-120.683967,San Luis Obispo County,Unincorporated,93446,CEC,NaN
2947,Stockton Acres,PG&E,pge,SUBSTATION,306249.0,60.0,37.963312,-121.323431,San Joaquin County,Stockton,95203,CEC,NaN
3018,Barrett,PG&E,pge,SUBSTATION,309938.0,60.0,37.939564,-122.347234,Contra Costa County,Richmond,94804,CEC,NaN


In [ ]:
PROCESSED

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

ROOT  = Path().resolve().parent
RAW   = ROOT / 'data' / 'processed' / 'substations'
PROC  = ROOT / 'data' / 'processed' / 'substations'

# Processed interchange (fromba always in CA8)
print('Loading processed substation load profiles ...')
loads = pd.read_csv(PROC / 'substation_load_profiles.csv', low_memory=False)
print(f'  {len(loads):,} load rows')
# Region TI
print('Loading locations data ...')
locations = pd.read_csv(RAW / 'substation_locations.csv')
print(f'  {len(locations):,} location rows')

In [ ]:
PROC

In [ ]:
foo = pd.read_csv(PROC / 'substation_attributs_clean.csv')
bar = pd.read_csv(PROC / 'substation_load_profiles_clean.csv')